In [ ]:
import pandas as pd
import os
import s3fs


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

s3_folder = "s3://lab/PESSD/csv"

files = fs.glob(f"{s3_folder}/*.csv")

for file in files:
    fs.get(file, "data/"+(file.removeprefix("lab/PESSD/csv/")))

In [8]:
df = pd.DataFrame()
for file in os.listdir("data") : 
    temp = pd.read_csv("data/"+file)
    df = pd.concat([df,temp])


In [21]:
BI = df[(df["audio_file"]=="BI_wav.wav") & ((df["start"]<(15*60))|((df["start"]>(17.5*60))&((df["stop"]<(89*60))))|(df["start"]>(92.5*60)))]

In [23]:
BIF = df[(df["audio_file"]=="BIF_wav.wav") & ((df["start"]<(12.5*60))|((df["start"]>(14.5*60))&((df["stop"]<(76*60)))))]

In [25]:
BMF = df[(df["audio_file"]=="BMF_wav.wav") & ((df["start"]>(1.5*60)))]

In [26]:
BMH = df[(df["audio_file"]=="BMH_wav.wav")]

In [28]:
SCR = df[(df["audio_file"]=="SCR2_wav.wav") & ((df["start"]<(28*60)))]

In [29]:
d = pd.concat([BI,BIF,BMH,BMF,SCR])

In [48]:
d2.to_csv("data/test2.csv")

In [1]:
!pip install bertopic sentence-transformers umap-learn hdbscan

In [2]:
import pandas as pd
d = pd.read_csv("data/test.csv")

In [ ]:
#Pas nécessaire
import spacy
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
d['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(d['text'])]

In [4]:
d

,Unnamed: 0,start,stop,text,main_speaker,main_g,audio_file
0,0,0.00,61.20,"En endroit, on est parti pour vivre encore un ...",SPEAKER_08,male,BI_wav.wav
1,1,61.20,103.60,"Oui, bonjour à tous, effectivement. Bienvenue ...",SPEAKER_02,male,BI_wav.wav
2,2,103.60,125.60,On imagine un Quentin qui aura envie de conser...,SPEAKER_12,male,BI_wav.wav
3,3,125.60,146.60,Moi je vais vous donner les numéros de Dossard...,SPEAKER_05,female,BI_wav.wav
4,4,146.60,150.60,Ouais l'équivalent à peu près de deux tours et...,SPEAKER_02,male,BI_wav.wav
...,...,...,...,...,...,...,...
1472,126,1611.68,1623.00,C'était raté sur la dernière course. La FP 4e ...,SPEAKER_11,male,SCR2_wav.wav
1473,127,1623.16,1633.56,"La Suède, qui revient à 3 secondes dès Nord-Vé...",SPEAKER_06,male,SCR2_wav.wav
1474,128,1634.88,1640.40,Ça va pas le mettre en confiance pour le début...,SPEAKER_10,male,SCR2_wav.wav
1475,129,1640.56,1645.64,C'était déjà la catastrophe à Nové-Mesto. Il a...,SPEAKER_06,female,SCR2_wav.wav


In [ ]:
# Install dependencies if needed
# pip install bertopic sentence-transformers umap-learn hdbscan

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base")

STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2
)
#umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=42)
seed =  [
["émotion","larme","ému","joie","triste", "histoire","hommage"],
["famille","parents","ami", "personnelle"],
["attaque","montée","effort","énergie","relance","sprint"],        # tactique
["médaille","podium","victoire","champion","record","victoire","classement","premier","deuxième","troisième"],              # résultats
["bravo","chapeau","incroyable","monstrueux","magnifique","énorme"],        # admiration
["tir","cible","pénalité","debout","couché","faute"],                      # technique
["âge","ans","carrière","champion du monde","olympique","saison"],
["rapide","forme","expérience","solide","rapide","bon"],          # profil athlète
["devant","revient","prend la tête","écart","seconde", "temps","tour","intermédiaire","course","piste"] 
]
# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)

# Fit the model
topics, probs = topic_model.fit_transform(d2["text"])

# Show discovered topics
print(topic_model.get_topic_info())

# Show keywords for a specific topic
print(topic_model.get_topic(0))

# Visualize topics
topic_model.visualize_topics()

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

#keybertopic, extrait phrases saillantes
#tester lda
#virer phrases nulles et refaire tourner
#pas virer -1 sauf si faible échantillon et checker
#checker si semi-superviser pour trouver thèmes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5443.10it/s]
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-16 11:44:12,445 - BERTopic - Embedding - Transforming documents to embeddings.
Batches:  57%|█████▋    | 26/46 [08:21<06:06, 18.34s/it]

In [98]:
t = pd.DataFrame({
    "text": d2['text'],
    "topic": topics,
    "topic_prob": probs.max(axis=1)  # max probability per text
})

In [99]:
t["topic"].value_counts()

topic
-1     508
 0      88
 1      86
 2      70
 3      49
 4      49
 5      44
 6      43
 7      41
 8      40
 9      35
 10     35
 11     34
 12     32
 13     32
 14     29
 15     28
 16     23
 17     21
 18     20
 19     20
 20     19
 21     17
 22     16
 23     15
 24     14
 25     13
 26     12
 27     11
Name: count, dtype: int64

In [67]:
topic_model.get_topic(4,full=True)

False

In [125]:
#1 : 4, 17, 18 retirés
#2 : 
for k in t[t["topic"]==26]["text"] : 
    print(k)

Oui, bonjour à tous, effectivement. Bienvenue à Antol, en Terre Selva. Moins de soleil qu'il y a deux jours, mais les Français, eh bien, sont au beau fixe. On est avec Marie Dorin, qui a été championne olympique multiple, championne du monde de Biathlon, et Frédéric Jean, qui était l'entraîneur de l'équipe de France féminine jusqu'à il y a 4 ans à Pékin. On était ensemble, avec vous Marie, au commentaire, et vous Fred, vous étiez dans l'encadrement de l'équipe de France, pour suivre le sacre de Quentin Fignon-Mayet. C'est vrai qu'aujourd'hui, les quatre Français peuvent jouer le podium et bien sûr la plus belle des médailles.
De retour à Antol Center Selva pour la première course individuelle de Biathlon. C'est 25e Jeu Olympique d'hiver. Jeux de Milan-Cortina. Vous n'avez rien manqué si vous nous rejoignez. Et bien, un seul français s'était lancé pour l'instant, Fabien Claude. Et bien, il nous fait du Fabien Claude, c'est-à-dire qu'il est en train de passer en tête aux intermédiaires.


In [28]:
mask = t[(t["topic"]!=4)&(t["topic"]!=17)&(t["topic"]!=18)]["text"]

In [29]:
d2 = d.merge(mask, on="text")

In [64]:
d2.merge(t,on="text")[d2.merge(t,on="text")["topic"]==18]

,Unnamed: 0,start,stop,text,main_speaker,main_g,audio_file,topic,topic_prob
498,398,4755.72,4757.72,C'est une belle histoire pour lui.,SPEAKER_05,female,BI_wav.wav,18,1.00000
588,515,6473.40,6476.36,"Il y a des leçons à tirer, malgré la déception...",SPEAKER_11,male,BI_wav.wav,18,1.00000
605,11,424.80,440.80,Juste le temps peut-être de saluer dans la raq...,SPEAKER_11,male,BIF_wav.wav,18,0.14816
937,342,4304.68,4312.68,C'est les belles histoires des jeux. Les belle...,SPEAKER_06,female,BIF_wav.wav,18,1.00000
948,353,4474.68,4496.68,"Oui, c'est vraiment... Bien évidemment, Trista...",SPEAKER_11,male,BIF_wav.wav,18,1.00000
1005,56,814.00,820.00,C'est long une Olympiade Avec toutes les émoti...,SPEAKER_04,male,BMH_wav.wav,18,1.00000
1041,97,1299.90,1307.90,La santé c'est le plus important. Elle peut êt...,SPEAKER_03,male,BMH_wav.wav,18,1.00000
1048,104,1369.90,1373.90,"Quand on connaît ses qualités, ça fend le cœur...",SPEAKER_04,female,BMH_wav.wav,18,1.00000
1621,173,1963.72,1972.56,"Va se laisser prendre par les émotions. Allez,...",SPEAKER_01,male,BMF_wav.wav,18,1.00000
1954,246,2668.36,2672.36,Oh les belles émotions.,SPEAKER_04,music,BMF_wav.wav,18,1.00000


In [48]:
topic_model.get_topic(-1)

[('tir', np.float64(0.019705384158766548)),
 ('course', np.float64(0.01810100205230608)),
 ('julia', np.float64(0.016166922702569482)),
 ('secondes', np.float64(0.01581559885555621)),
 ('petit', np.float64(0.015216028128888991)),
 ('vraiment', np.float64(0.014750896397992897)),
 ('faut', np.float64(0.014264980364740047)),
 ('allez', np.float64(0.014103260774615873)),
 ('20', np.float64(0.01389915649281138)),
 ('aller', np.float64(0.013604787866193726))]

In [132]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


sentence_embeddings = model.encode(d2["text"])
theme_embeddings = model.encode(themes)

sim = cosine_similarity(sentence_embeddings, theme_embeddings)
labels = sim.argmax(axis=1)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5220.97it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [136]:
d2["labels"]=labels

In [147]:
d2[d2["labels"]==8]["text"]

0       En endroit, on est parti pour vivre encore un ...
3       Moi je vais vous donner les numéros de Dossard...
4       Ouais l'équivalent à peu près de deux tours et...
7       Il a un rôle important dans cette équipe. Donc...
9       On discutait hier justement dans la reconnaiss...
                              ...                        
1427    Et on prend le temps de savourer. Et nous auss...
1428    Elle vient de passer derrière le pas de tir. E...
1434    Simon l'a dit, Simon Fourcade, l'entraîneur de...
1440    La Suède, qui revient à 3 secondes dès Nord-Vé...
1441    Ça va pas le mettre en confiance pour le début...
Name: text, Length: 444, dtype: object

In [131]:
themes = []
for i in seed : 
    themes.append(" ".join(i))
themes

['émotion larme ému joie triste histoire hommage',
 'famille parents ami personnelle',
 'attaque montée effort énergie relance sprint',
 'médaille podium victoire champion record victoire classement premier deuxième troisième',
 'bravo chapeau incroyable monstrueux magnifique énorme',
 'tir cible pénalité debout couché faute',
 'âge ans carrière champion du monde olympique saison',
 'rapide forme expérience solide rapide bon',
 'devant revient prend la tête écart seconde temps tour intermédiaire course piste']

In [65]:
test = topic.merge(d, on="text")

In [67]:
test[(test["topic"]==10)|(test["topic"]==15)]

,text,topic,topic_prob,start,stop,main_speaker,main_g,audio_file
8,Pour vous en dire un petit peu plus sur la pis...,10,1.000000,289.60,345.80,SPEAKER_05,female,BI_wav.wav
9,On discutait hier justement dans la reconnaiss...,10,0.124827,345.80,356.40,SPEAKER_12,male,BI_wav.wav
11,Après c'est vrai que cette année il y a des st...,15,1.000000,374.20,386.60,SPEAKER_05,female,BI_wav.wav
13,"C'est un athlète solide, le jeune Suisse qui p...",15,0.238053,421.20,434.20,SPEAKER_12,male,BI_wav.wav
25,C'est un athlète qui a énormément d'expérience...,15,0.219121,604.20,622.20,SPEAKER_12,male,BI_wav.wav
26,"Cette année, il est un tout petit peu moins bo...",15,1.000000,622.20,634.20,SPEAKER_05,female,BI_wav.wav
28,"Fabien, c'est vraiment un skieur qui fait part...",15,1.000000,677.20,709.20,SPEAKER_05,female,BI_wav.wav
78,"Là, ça fait mal. On voit dans le regard qu'il ...",10,0.364798,1586.20,1595.20,SPEAKER_12,female,BI_wav.wav
93,On a l'impression qu'il y a les jambes qui tre...,10,1.000000,1724.20,1734.20,SPEAKER_05,female,BI_wav.wav
639,"Comme on l'avait dit pour le relais mixte, c'e...",10,0.152644,4025.84,4036.72,SPEAKER_05,female,BI_wav.wav
